In [1]:
from Functions import *
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

In [2]:
# Load the Excel files

df_1_DATA = load_excel(r"D:\Projects\Lux_Project_Intern\Data\1_DATA.xlsx")
df_2_LIST_OF_COLUMNS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\2_LIST_OF_COLUMNS.xlsx")
df_3_LOB = load_excel(r"D:\Projects\Lux_Project_Intern\Data\3_LOB.xlsx")
df_4_TESTS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\4_TESTS.xlsx")
df_5_FINANCIALS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\5_FINANCIALS.xlsx")

In [3]:
# Clean and align the columns of df_1_DATA based on the column names in df_2_LIST_OF_COLUMNS

df_1_DATA = clean_and_align_columns(df_1_DATA, df_2_LIST_OF_COLUMNS['Column Names'].tolist())

In [4]:
# clean LA_LOB, LA_STATUS columns from df_1_DATA

df_1_DATA = clean_lob_status(
    df_1_DATA,
    lob_col="LA_LOB",
    status_col="LA_STATUS",
    lob_choices=df_3_LOB["Line of business"].dropna().tolist(),
    status_choices=df_3_LOB["Status"].dropna().tolist(),
    threshold=75
)

## #Data tests 

In [5]:
# Example usage:
df_1_DATA, report_df, issue_rows = MASTER_DATA_QUALITY_REPORT(
    df1=df_1_DATA,
    # pass None if you don't have this sheet
    category_col='TESTING_CATEGORY',
    loss_date_col='LA_LOSS_DATE',
    as_at_col='LA_AS_AT_DATE',
    payment_date_col='LA_PAYMENT_DATE',
    status_col='LA_STATUS',
    condition_value='PAID',
    key_col='POLICY_NUMBER',
    valuation_date_col='VALUATION_DATE'
)
report_df


Rows with status != 'PAID' (payment date wiped): 1
Rows with status == 'PAID' but missing payment date: 2
Rows checked (both dates present): 36
Rows skipped (missing date(s)): 47
Invalid rows (payment date before loss date): 0
DATA QUALITY REPORT
                           step                          column          status                                                                                        detail
       check_missing (category)                TESTING_CATEGORY           ERROR                                              Column 'TESTING_CATEGORY' not found in dataframe
      check_missing (loss date)                    LA_LOSS_DATE            PASS                                                                    No nulls in 'LA_LOSS_DATE'
               AS_AT_DATE_CLEAN                   LA_AS_AT_DATE PARTIALLY_FIXED 4 nulls found in 'LA_AS_AT_DATE' -> 4 rows substituted using quarter-end of 'LA_PAYMENT_DATE'
             PAYMENT_DATE_CLEAN                 LA_PAYMEN

,step,column,status,detail
0,check_missing (category),TESTING_CATEGORY,ERROR,Column 'TESTING_CATEGORY' not found in dataframe
1,check_missing (loss date),LA_LOSS_DATE,PASS,No nulls in 'LA_LOSS_DATE'
2,AS_AT_DATE_CLEAN,LA_AS_AT_DATE,PARTIALLY_FIXED,4 nulls found in 'LA_AS_AT_DATE' -> 4 rows sub...
3,PAYMENT_DATE_CLEAN,LA_PAYMENT_DATE,INFO,1 dates wiped (status != 'PAID'); 2 PAID rows ...
4,PAYMENT_VS_LOSS_DATE_VALIDATE,LA_PAYMENT_DATE vs LA_LOSS_DATE,PASS,0 rows have payment date before loss date
5,VALUATION_VS_LOSS_DATE_VALIDATE,-,SKIPPED,df2 / key_col / valuation_date_col not provided
